In [ ]:
import pandas as pd

# 1. URL do arquivo no GitHub (formato Raw)
url = "https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Sketch/Data_cleaning/Tabelas/Tabela_Evasao_2024.csv"

# Carregar o dataset
df = pd.read_csv(url)

# ==============================================================================
# 2. CRIAÇÃO DAS MÉTRICAS GERAIS
# ==============================================================================
# Captura a maior taxa de evasão registrada na escola (seja fundamental ou médio)
df['evasao_geral'] = df[['evasao_fundamental_total', 'evasao_medio_total']].max(axis=1)

# NOVA MÉTRICA: Captura a maior taxa de reprovação registrada na escola
df['reprovacao_geral'] = df[['reprovacao_fundamental_total', 'reprovacao_medio_total']].max(axis=1)

# Remover do cálculo escolas que não possuem nenhum dado de evasão preenchido
df_valid = df.dropna(subset=['evasao_geral'])

# 3. Ordenação dos dados (Top maiores evasões por estado e tipo)
# Nota: Se quiseres ordenar pelas maiores reprovações, basta mudar 'evasao_geral' para 'reprovacao_geral' abaixo
df_sorted = df_valid.sort_values(
    by=['uf', 'dependencia_adm', 'evasao_geral'],
    ascending=[True, True, False]
)

# 4. Pegar as 3 primeiras de cada combinação de UF e Tipo
df_top3 = df_sorted.groupby(['uf', 'dependencia_adm']).head(3).copy()

# ==============================================================================
# TRATAMENTO PARA EXPORTAÇÃO
# ==============================================================================
# Adicionamos as colunas de reprovação (as originais e a nova geral) no relatório
colunas_relatorio = [
    'uf', 'dependencia_adm', 'codigo_escola', 'nome_escola',
    'reprovacao_fundamental_total', 'reprovacao_medio_total', 'reprovacao_geral',
    'evasao_fundamental_total', 'evasao_medio_total', 'evasao_geral'
]
df_relatorio = df_top3[colunas_relatorio].copy()

# Lista de todas as colunas originais que podem conter NaNs para tratamento
colunas_taxas_originais = [
    'reprovacao_fundamental_total', 'reprovacao_medio_total',
    'evasao_fundamental_total', 'evasao_medio_total'
]

# Substitui os NaNs das colunas originais por "Não informado"
for coluna in colunas_taxas_originais:
    df_relatorio[coluna] = df_relatorio[coluna].fillna("Não informado")

# Tratamento para as colunas gerais calculadas:
# Se a escola não tinha dados em nenhuma das duas colunas de reprovação, a 'reprovacao_geral' será NaN.
# Vamos garantir que ela também apareça como "Não informado".
df_relatorio['reprovacao_geral'] = df_relatorio['reprovacao_geral'].fillna("Não informado")

# Força o código da escola a ser tratado como string
df_relatorio['codigo_escola'] = df_relatorio['codigo_escola'].astype(str)

# 5. EXPORTAR PARA UM NOVO ARQUIVO CSV
df_relatorio.to_csv('maiores_taxas_evasao_e_reprovacao_2024.csv', index=False, encoding='utf-8-sig')

print("📊 Análise concluída! O arquivo 'maiores_taxas_evasao_e_reprovacao_2024.csv' foi gerado com sucesso.")
print("Verifica a barra lateral esquerda (ícone de pasta 📁) para descarregar o ficheiro.")